In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
import keras_tuner as kt
import tensorboard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Input,SimpleRNN
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

In [2]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [3]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [4]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [5]:
SEQ_LEN = 24
HORIZON = 24

X_train, y_train = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val, y_val     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test, y_test   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape:  {X_test.shape}")

X_train Shape: (101593, 24, 13)
X_test Shape:  (21733, 24, 13)


In [66]:
X_train[0:1]

array([[[0.30628884, 0.        , 0.        , 0.04347826, 0.33333333,
         0.26666667, 0.01923077, 0.        , 0.34549668, 0.31393658,
         0.28604235, 0.4565556 , 0.2340761 ],
        [0.2867376 , 0.        , 0.        , 0.08695652, 0.33333333,
         0.26666667, 0.01923077, 0.        , 0.30628884, 0.29760876,
         0.27163173, 0.45609663, 0.23637708],
        [0.27989045, 0.        , 0.        , 0.13043478, 0.33333333,
         0.26666667, 0.01923077, 0.        , 0.2867376 , 0.29139366,
         0.26876646, 0.45544421, 0.24015315],
        [0.27841567, 0.        , 0.        , 0.17391304, 0.33333333,
         0.26666667, 0.01923077, 0.        , 0.27989045, 0.29491204,
         0.27365427, 0.45475385, 0.24429918],
        [0.28998209, 0.        , 0.        , 0.2173913 , 0.33333333,
         0.26666667, 0.01923077, 0.        , 0.27841567, 0.31006004,
         0.2920257 , 0.45376383, 0.25009537],
        [0.32918993, 0.        , 0.        , 0.26086957, 0.33333333,
         0.

In [8]:
y_test_mw = scaler.inverse_transform(y_test.reshape(-1, 1))

In [30]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [33]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [ ]:
#rnn model

In [ ]:
#drop out 0.3 , 0.5

In [6]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0102 - mae: 0.0687 - val_loss: 0.0025 - val_mae: 0.0379
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0029 - mae: 0.0411 - val_loss: 0.0023 - val_mae: 0.0358
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0026 - mae: 0.0384 - val_loss: 0.0021 - val_mae: 0.0346
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0024 - mae: 0.0374 - val_loss: 0.0022 - val_mae: 0.0352
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0024 - mae: 0.0367 - val_loss: 0.0020 - val_mae: 0.0334
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0023 - mae: 0.0362 - val_loss: 0.0019 - val_mae: 0.0329
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0022 - mae: 0.0358 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0022 - mae: 0.0354 - val_loss: 0.0020 - val_mae: 0.0328
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [9]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1545.17 MW
RMSE: 2137.41 MW
MAPE: 4.84%
R2:   0.8899


In [ ]:
#drop out 0.5

In [10]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0139 - mae: 0.0808 - val_loss: 0.0032 - val_mae: 0.0442
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0036 - mae: 0.0464 - val_loss: 0.0022 - val_mae: 0.0351
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0032 - mae: 0.0432 - val_loss: 0.0022 - val_mae: 0.0354
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0030 - mae: 0.0420 - val_loss: 0.0021 - val_mae: 0.0340
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0029 - mae: 0.0414 - val_loss: 0.0022 - val_mae: 0.0348
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0029 - mae: 0.0408 - val_loss: 0.0021 - val_mae: 0.0341
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0028 - mae: 0.0400 - val_loss: 0.0024 - val_mae: 0.0362
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0027 - mae: 0.0395 - val_loss: 0.0021 - val_mae: 0.0346
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/st

In [11]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1626.73 MW
RMSE: 2192.86 MW
MAPE: 5.20%
R2:   0.8842


In [ ]:
#0.2

In [13]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0088 - mae: 0.0631 - val_loss: 0.0023 - val_mae: 0.0368
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0026 - mae: 0.0384 - val_loss: 0.0020 - val_mae: 0.0335
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0023 - mae: 0.0359 - val_loss: 0.0021 - val_mae: 0.0347
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0022 - mae: 0.0348 - val_loss: 0.0021 - val_mae: 0.0353
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0021 - mae: 0.0343 - val_loss: 0.0020 - val_mae: 0.0338
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0020 - mae: 0.0337 - val_loss: 0.0019 - val_mae: 0.0316
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0020 - mae: 0.0333 - val_loss: 0.0018 - val_mae: 0.0305
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0019 - mae: 0.0329 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [14]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1552.73 MW
RMSE: 2112.70 MW
MAPE: 4.97%
R2:   0.8925


In [ ]:
model_rnn.save(r'../models/drop_ou_rnn.keras')

In [17]:
history_df =pd.DataFrame(history_rnn.history)

history_df.to_csv(r'../log/dropout_rnn.csv',index=False)

In [ ]:
#batch normalization

In [20]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 0.0484 - mae: 0.1413 - val_loss: 0.0103 - val_mae: 0.0801
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0093 - mae: 0.0750 - val_loss: 0.0047 - val_mae: 0.0541
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0047 - mae: 0.0531 - val_loss: 0.0033 - val_mae: 0.0433
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0037 - mae: 0.0468 - val_loss: 0.0028 - val_mae: 0.0404
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0033 - mae: 0.0439 - val_loss: 0.0023 - val_mae: 0.0361
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0029 - mae: 0.0416 - val_loss: 0.0024 - val_mae: 0.0384
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0027 - mae: 0.0399 - val_loss: 0.0027 - val_mae: 0.0400
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0026 - mae: 0.0386 - val_loss: 0.0021 - val_mae: 0.0347
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [21]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE:  1635.97 MW
RMSE: 2206.60 MW
MAPE: 5.19%
R2:   0.8827


In [ ]:
#adam

In [22]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0091 - mae: 0.0617 - val_loss: 0.0029 - val_mae: 0.0414
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0022 - mae: 0.0354 - val_loss: 0.0022 - val_mae: 0.0349
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0018 - mae: 0.0315 - val_loss: 0.0020 - val_mae: 0.0326
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0017 - mae: 0.0304 - val_loss: 0.0018 - val_mae: 0.0313
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0294 - val_loss: 0.0018 - val_mae: 0.0310
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0016 - mae: 0.0288 - val_loss: 0.0018 - val_mae: 0.0306
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0015 - mae: 0.0285 - val_loss: 0.0017 - val_mae: 0.0301
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0015 - mae: 0.0280 - val_loss: 0.0017 - val_mae: 0.0303
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/st

In [23]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1486.33 MW
RMSE: 2049.95 MW
MAPE: 4.76%
R2:   0.8988


In [24]:
model_rnn.save(r'../models/adam_rnn.keras')
history_df =pd.DataFrame(history_rnn.history)

history_df.to_csv(r'../log/adam_rnn.csv',index=False)

In [ ]:
#sgd

In [25]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='sgd', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0327 - mae: 0.1308 - val_loss: 0.0159 - val_mae: 0.1010
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0149 - mae: 0.0945 - val_loss: 0.0133 - val_mae: 0.0915
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0126 - mae: 0.0872 - val_loss: 0.0115 - val_mae: 0.0850
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0108 - mae: 0.0813 - val_loss: 0.0101 - val_mae: 0.0795
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0097 - mae: 0.0774 - val_loss: 0.0092 - val_mae: 0.0758
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0091 - mae: 0.0749 - val_loss: 0.0086 - val_mae: 0.0735
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0086 - mae: 0.0730 - val_loss: 0.0082 - val_mae: 0.0717
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0082 - mae: 0.0714 - val_loss: 0.0079 - val_mae: 0.0701
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/st

In [26]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  3315.68 MW
RMSE: 4207.07 MW
MAPE: 10.92%
R2:   0.5736


In [ ]:
#rms prop

In [27]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='RMSprop', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,verbose=1)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0057 - mae: 0.0536 - val_loss: 0.0026 - val_mae: 0.0390
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0022 - mae: 0.0356 - val_loss: 0.0025 - val_mae: 0.0380
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0020 - mae: 0.0329 - val_loss: 0.0021 - val_mae: 0.0341
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0018 - mae: 0.0314 - val_loss: 0.0021 - val_mae: 0.0335
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0017 - mae: 0.0305 - val_loss: 0.0020 - val_mae: 0.0328
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0017 - mae: 0.0298 - val_loss: 0.0019 - val_mae: 0.0320
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0292 - val_loss: 0.0022 - val_mae: 0.0352
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0288 - val_loss: 0.0018 - val_mae: 0.0307
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/st

In [28]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE:  1534.33 MW
RMSE: 2114.51 MW
MAPE: 4.91%
R2:   0.8923


In [ ]:
#learning rate

In [34]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,callbacks=[lr_scheduler])

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0055 - mae: 0.0496 - val_loss: 0.0023 - val_mae: 0.0359 - learning_rate: 0.0010
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0019 - mae: 0.0327 - val_loss: 0.0020 - val_mae: 0.0331 - learning_rate: 0.0010
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0018 - mae: 0.0310 - val_loss: 0.0019 - val_mae: 0.0313 - learning_rate: 0.0010
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0017 - mae: 0.0300 - val_loss: 0.0018 - val_mae: 0.0308 - learning_rate: 0.0010
Epoch 5/10
1572/1588 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0016 - mae: 0.0294
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0016 - mae: 0.0294 - val_loss: 0.0018 - val_mae: 0.0305 - learning_rate: 0.0010
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0017 - val_mae: 0.0298 - learning_r

In [35]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1464.36 MW
RMSE: 2037.44 MW
MAPE: 4.68%
R2:   0.9000


In [36]:
model_rnn.save(r'../models/lr_rnn.keras')
history_df =pd.DataFrame(history_rnn.history)

history_df.to_csv(r'../log/lr_rnn.csv',index=False)

In [ ]:
#early stopping

In [37]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64,callbacks=[early_stop])

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0058 - mae: 0.0498 - val_loss: 0.0035 - val_mae: 0.0474
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0020 - mae: 0.0330 - val_loss: 0.0021 - val_mae: 0.0333
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0296 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0290 - val_loss: 0.0022 - val_mae: 0.0355
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0016 - mae: 0.0288 - val_loss: 0.0018 - val_mae: 0.0301
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0015 - mae: 0.0283 - val_loss: 0.0017 - val_mae: 0.0301
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0018 - val_mae: 0.0299
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/st

In [38]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE:  1463.41 MW
RMSE: 2034.92 MW
MAPE: 4.67%
R2:   0.9002


In [39]:
model_rnn.save(r'../models/early_rnn.keras')
history_df =pd.DataFrame(history_rnn.history)

history_df.to_csv(r'../log/early_rnn.csv',index=False)

In [ ]:
#batch size

In [40]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=16)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6350/6350 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 0.0034 - mae: 0.0411 - val_loss: 0.0021 - val_mae: 0.0344
Epoch 2/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0018 - mae: 0.0316 - val_loss: 0.0019 - val_mae: 0.0319
Epoch 3/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0017 - mae: 0.0298 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 4/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0016 - mae: 0.0289 - val_loss: 0.0016 - val_mae: 0.0285
Epoch 5/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0015 - mae: 0.0283 - val_loss: 0.0017 - val_mae: 0.0305
Epoch 6/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0019 - val_mae: 0.0316
Epoch 7/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0014 - mae: 0.0275 - val_loss: 0.0016 - val_mae: 0.0295
Epoch 8/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0014 - mae: 0.0272 - val_loss: 0.0017 - val_mae: 0.0286
Epoch 9/10
6350/6350 ━━━━━━━━━━━━━━━━━━━━ 1

In [41]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1493.23 MW
RMSE: 2083.45 MW
MAPE: 4.79%
R2:   0.8954


In [ ]:
#batch size 32

In [42]:
input_shape = (X_train.shape[1], X_train.shape[2])

# Build Basic RNN
model_rnn = Sequential([
    SimpleRNN(64, return_sequences=False, input_shape=input_shape),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=32)

Epoch 1/10


c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3175/3175 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.0043 - mae: 0.0439 - val_loss: 0.0020 - val_mae: 0.0329
Epoch 2/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0018 - mae: 0.0317 - val_loss: 0.0019 - val_mae: 0.0310
Epoch 3/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0017 - mae: 0.0301 - val_loss: 0.0019 - val_mae: 0.0315
Epoch 4/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0016 - mae: 0.0292 - val_loss: 0.0017 - val_mae: 0.0306
Epoch 5/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0015 - mae: 0.0284 - val_loss: 0.0017 - val_mae: 0.0292
Epoch 6/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0017 - val_mae: 0.0294
Epoch 7/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - loss: 0.0014 - mae: 0.0274 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 8/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.0014 - mae: 0.0272 - val_loss: 0.0017 - val_mae: 0.0315
Epoch 9/10
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/s

In [43]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
MAE:  1546.34 MW
RMSE: 2115.80 MW
MAPE: 4.98%
R2:   0.8922


In [ ]:
#number of layers

In [44]:
input_shape = (X_train.shape[1], X_train.shape[2])

model_rnn = Sequential([
    SimpleRNN(64, return_sequences=True,input_shape=input_shape),
    Dropout(0.2),

    SimpleRNN(32),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_rnn = model_rnn.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 0.0126 - mae: 0.0768 - val_loss: 0.0030 - val_mae: 0.0432
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0033 - mae: 0.0438 - val_loss: 0.0022 - val_mae: 0.0360
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0028 - mae: 0.0402 - val_loss: 0.0021 - val_mae: 0.0349
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0026 - mae: 0.0388 - val_loss: 0.0020 - val_mae: 0.0332
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0025 - mae: 0.0378 - val_loss: 0.0020 - val_mae: 0.0329
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0024 - mae: 0.0372 - val_loss: 0.0019 - val_mae: 0.0322
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0023 - mae: 0.0367 - val_loss: 0.0020 - val_mae: 0.0331
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0023 - mae: 0.0363 - val_loss: 0.0019 - val_mae: 0.0320
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━

In [45]:
y_pred_bilstm_scaled = model_rnn.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE:  1573.88 MW
RMSE: 2152.38 MW
MAPE: 4.97%
R2:   0.8884


In [ ]:
#hyper tuning

In [54]:
input_shape = (X_train.shape[1], X_train.shape[2])

def build_model(hp):

    model = Sequential([
        Input(shape=input_shape),

        SimpleRNN(
            units=hp.Choice(
                "rnn_units",
                [32, 64, 128]
            )
        ),

        BatchNormalization(),

        Dense(
            units=hp.Choice(
                "dense_units",
                [16, 32, 64]
            ),
            activation="relu"
        ),

        Dropout(
            hp.Choice(
                "dropout",
                [0.2, 0.3, 0.5]
            )
        ),

        Dense(HORIZON)
    ])

    model.compile(
         optimizer=hp.Choice(
        "optimizer",
        values=["adam", "rmsprop", "sgd"]),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [55]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=5,
    directory="rnn_tuning",
    project_name="rnn_forecasting"
)

Reloading Tuner from rnn_tuning\rnn_forecasting\tuner0.json


In [56]:
tuner.search(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Trial 5 Complete [00h 01m 24s]
val_loss: 0.003083763876929879

Best val_loss So Far: 0.0018669947749003768
Total elapsed time: 00h 07m 35s


In [57]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("RNN units:", best_hp.get("rnn_units"))
print("Dense units:", best_hp.get("dense_units"))
print("Dropout:", best_hp.get("dropout"))

RNN units: 32
Dense units: 64
Dropout: 0.3


In [58]:
best_model = tuner.get_best_models(num_models=1)[0]

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\saving\saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(store)


In [59]:
history_rnn = best_model.fit(X_train, y_train,validation_data=(X_val, y_val),epochs=10,batch_size=64)

Epoch 1/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - loss: 0.0029 - mae: 0.0406 - val_loss: 0.0018 - val_mae: 0.0317
Epoch 2/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0028 - mae: 0.0404 - val_loss: 0.0019 - val_mae: 0.0321
Epoch 3/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0028 - mae: 0.0403 - val_loss: 0.0018 - val_mae: 0.0311
Epoch 4/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0028 - mae: 0.0400 - val_loss: 0.0018 - val_mae: 0.0312
Epoch 5/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0028 - mae: 0.0399 - val_loss: 0.0019 - val_mae: 0.0325
Epoch 6/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0027 - mae: 0.0397 - val_loss: 0.0019 - val_mae: 0.0338
Epoch 7/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0027 - mae: 0.0395 - val_loss: 0.0018 - val_mae: 0.0306
Epoch 8/10
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.0027 - mae: 0.0396 - val_loss: 0.0017 - val_mae: 0.0302
Epoch 9/10
1588/1588 ━━━━━━━━━━━━━━━━━━━

In [60]:
y_pred_bilstm_scaled = best_model.predict(X_test)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

680/680 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MAE:  1559.91 MW
RMSE: 2150.08 MW
MAPE: 4.90%
R2:   0.8886
